In [1]:
%load_ext autoreload
%autoreload 2


In [2]:
import os
from typing import Union

from google.cloud import secretmanager

# set the Google Cloud project if env is set, otherwise default to rxrx-medchem-auto-dev
google_cloud_project = os.getenv("GOOGLE_CLOUD_PROJECT", "rxrx-medchem-auto-dev")


def access_secret_version(
    secret_name: str,
    project_id: str = google_cloud_project,
    secret_ver: Union[int, str] = "latest",  # noqa
) -> str:
    """
    Accesses a secret version from Google Cloud Secret Manager.

    Args:
        secret_name (str): The name of the secret.
        project_id (str, optional): The project ID. Defaults to GOOGLE_CLOUD_PROJECT env var or "rxrx-medchem-auto-dev".

        secret_ver (Union[int, str], optional): The version of the secret to access. Defaults to "latest".

    Returns:
        str: The secret data.

    Examples:
        # Access the latest version of the secret
        >>> access_secret_version("my_secret")
        'my_secret_data'

        # Access a specific version of the secret
        >>> access_secret_version("my_secret", secret_ver=2)
        'my_secret_data_v2'
    """
    client = secretmanager.SecretManagerServiceClient()
    name = f"projects/{project_id}/secrets/{secret_name}/versions/{secret_ver}"
    response = client.access_secret_version(request={"name": name})
    data = response.payload.data.decode("UTF-8")
    assert data is not None, "Secret not found"  # noqa
    return data


def set_env_secrets(secret_list: list[str]):
    for secret in secret_list:
        secret_data = access_secret_version(secret)
        os.environ[secret] = secret_data


def set_env_secrets_safely(secret_list: list[str]):
    from loguru import logger

    try:
        set_env_secrets(["OPENAI_API_KEY", "LANGCHAIN_API_KEY"])
    except Exception as e:
        import os
        import sys

        logger.error(f"Error setting environment secrets {e}")
        is_in_ci = os.getenv("CF_REPO_NAME", None)
        if not is_in_ci:
            sys.exit(1)

In [3]:
set_env_secrets_safely(["OPENAI_API_KEY", "LANGCHAIN_API_KEY"])

In [4]:
import os
from typing import Union

from aiolimiter import AsyncLimiter
from elasticsearch import AsyncElasticsearch


class NoopLimiter:
    """A no-op rate limiter for when rate limiting is not needed."""

    def __init__(self):
        pass

    async def __aenter__(self):
        return self

    async def __aexit__(self, exc_type, exc, tb):
        pass

    def locked(self):
        return False

    async def acquire(self):
        return True

    def release(self):
        pass

    def get_value(self):
        return float("inf")


class FulltextIndex:
    """Basic search client for Elasticsearch"""

    def __init__(
        self,
        url=os.environ.get(
            "FULLTEXT_ELASTICSEARCH_URL",
            "https://elasticsearch.centaur-platform-dev.com:443",
        ),
        index=os.environ.get("FULLTEXT_ELASTICSEARCH_INDEX", "full-text-articles"),
        rate_limit: Union[int, None] = None,
    ):
        self.__client = AsyncElasticsearch(
            hosts=[url],
            max_retries=20,
            retry_on_timeout=True,
            request_timeout=120,
        )
        self.__index = index
        self.__rate_limiter = AsyncLimiter(rate_limit) if isinstance(rate_limit, int) else NoopLimiter()

    async def close(self):
        await self.__client.close()

    async def __aenter__(self):
        return self

    async def __aexit__(self, *execinfo):
        await self.__client.close()

    async def search(self, body):
        async with self.__rate_limiter:
            return await self.__client.search(index=self.__index, **body)

In [5]:

async def create_search_body(keywords: list[str], top_k: int = 50) -> dict:
    """Create ElasticSearch query body with improved relevance scoring."""
    # Join keywords for phrase matching
    keywords_str = " ".join(keywords)

    search_body = {
        "query": {
            "bool": {
                "should": [
                    {"match": {"title": {"query": keywords_str, "boost": 3}}},
                    {"match": {"abstract": {"query": keywords_str, "boost": 2}}},
                    {"match": {"body": {"query": keywords_str, "boost": 5}}},
                ]
            }
        },
        "highlight": {
            "fields": {
                "body": {},
            },
            "pre_tags": ["<em>"],
            "post_tags": ["</em>"],
        },
        "size": top_k,
    }
    return search_body


In [6]:
from tenacity import (
    retry,
    retry_if_exception_type,
    stop_after_attempt,
    wait_exponential,
)

MAX_RETRIES = 2
BASE_DELAY = 1  # Initial delay in seconds
MAX_DELAY = 3  # Maximum delay in seconds
from loguru import logger


In [7]:
@retry(
    stop=stop_after_attempt(MAX_RETRIES),
    wait=wait_exponential(multiplier=BASE_DELAY, max=MAX_DELAY),
    retry=retry_if_exception_type(Exception),
    reraise=True,
)
async def _search_with_retry(client, search_body):
    """Perform elasticsearch search with retry logic."""
    return await client.search(body=search_body)


async def retrieve_articles(search_body: dict) -> list[dict]:
    """Retrieve articles from ElasticSearch using the search body."""
    retrieved_articles = []

    try:
        async with FulltextIndex() as client:
            try:
                # Use the retry-wrapped search function
                response = await _search_with_retry(client, search_body)

                hits = response.get("hits", {}).get("hits", [])
                logger.debug(f"Retrieved {len(hits)} hits")

                for _, hit in enumerate(hits):
                    source = hit.get("_source", {})
                    paper_id = hit.get("_id", "unknown_id")

                    title = source.get("title", "No title provided")

                    abstract = ""
                    for section in source.get("sections", []):
                        if section.get("type") == "ABSTRACT":
                            abstract = section.get("text", "")
                            break

                    main_content = []
                    for section in source.get("sections", []):
                        section_type = section.get("type", "").upper()
                        section_title = section.get("heading", "").upper()

                        # Skip sections that are not relevant
                        if any(
                            term in section_type or term in section_title
                            for term in [
                                "ABSTRACT",
                                "ACKNOWLEDGMENT",
                                "REFERENCE",
                                "FUNDING",
                                "AUTHOR",
                                "CONFLICT",
                                "DISCLOSURE",
                                "APPENDIX",
                            ]
                        ):
                            continue

                        # Add the section text
                        section_text = section.get("text", "")
                        if section_text:
                            main_content.append(section_text)

                    content = " ".join(main_content)

                    retrieved_articles.append(
                        {
                            "id": paper_id,
                            "title": title,
                            "abstract": abstract,
                            "content": content,
                            "score": hit.get("_score", 0),
                        }
                    )

                paper_ids = [article["id"] for article in retrieved_articles]
                logger.debug(f"Retrieved paper IDs: {paper_ids}")

            except Exception as e:
                logger.error(f"Error performing search: {e}")
                logger.exception("Full exception details:")
    except Exception as e:
        logger.error(f"Error connecting to Elasticsearch: {e}")
        logger.exception("Full exception details:")

    # If no articles were found, add a fallback article
    if not retrieved_articles:
        logger.warning("No articles found. Adding fallback article.")
        retrieved_articles.append(
            {
                "id": "fallback_id",
                "title": "No relevant articles found",
                "abstract": "The search did not return any relevant scientific articles for the given query.",
                "content": "",
                "score": 0,
            }
        )

    return retrieved_articles

In [8]:
body = await create_search_body(["EGFR", "Imatinib"])

In [9]:
results = await retrieve_articles(body)

2025-08-15 16:19:17.428 | ERROR    | __main__:retrieve_articles:79 - Error performing search: BadRequestError(400, 'media_type_header_exception', 'Invalid media-type value on headers [Accept, Content-Type]', Accept version must be either version 8 or 7, but found 9. Accept=application/vnd.elasticsearch+json; compatible-with=9)
2025-08-15 16:19:17.429 | ERROR    | __main__:retrieve_articles:80 - Full exception details:
Traceback (most recent call last):

  File "/Users/emmanuel.noutahi/miniconda3/envs/dev/lib/python3.12/runpy.py", line 198, in _run_module_as_main
    return _run_code(code, main_globals, None,
           │         │     └ {'__name__': '__main__', '__doc__': 'Entry point for launching an IPython kernel.\n\nThis is separate from the ipykernel pack...
           │         └ <code object <module> at 0x100931b30, file "/Users/emmanuel.noutahi/Code/hooke-explain/.venv/lib/python3.12/site-packages/ipy...
           └ <function _run_code at 0x1009ae0c0>
  File "/Users/emmanuel.n